# Project 2 : Spark Data Quality Checks & NFL Analysis

**ST 554 : Analysis of Big Data with Python**  
**Author:** David Pressley

---

## Overview

This project explores two aspects of working with PySpark:

1. **Part I**: Design and implement a `SparkDataCheck` class that wraps a Spark SQL DataFrame and provides methods for data validation (bounds checking, level checking, null detection) and summarization (numeric min/max, string counts), then demonstrate the class on the air quality dataset.

2. **Part II**: Analyze weekly NFL data using both the **pandas-on-Spark** API and the **Spark SQL** DataFrame API, comparing quarterback stats across seasons 2005–2023.

---

## Part I : SparkDataCheck Class

### Setup

Start a Spark session and import our SparkDataCheck module.

In [17]:
from pyspark.sql import SparkSession
import pandas as pd

# Start Spark session
spark = SparkSession.builder.appName("p2").getOrCreate()

26/03/28 16:49:08 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [18]:
# Import our custom module (use importlib.reload if you update the .py file)
import importlib
import spark_data_check
importlib.reload(spark_data_check)
from spark_data_check import SparkDataCheck

### Read in the Air Quality Data (CSV method)

We use our `from_csv` class method to create an instance of `SparkDataCheck` from the air quality CSV.

In [19]:
import urllib.request

url = "https://www4.stat.ncsu.edu/online/datasets/air.csv"
urllib.request.urlretrieve(url, "air.csv")
air_check = SparkDataCheck.from_csv(spark, "air.csv")

### Demonstration of Validation Methods

Below we show 4–5 examples of each validation method, including edge cases.

#### Check Range Validation
These validation checks test specified use cases and edge cases for upper and lower bound range checks, and populate a newly created column with a boolean value asserting the value is in the range (or not)

In [39]:
# check_range examples

print("1. Both bounds on a numeric column (Temp between 0 and 40)")
air_check.check_range("T", lower=0, upper=40).df.show(5)

print("2. Only lower bound (Benzene Ground Truth >= 0)")
air_check = SparkDataCheck.from_csv(spark, "air.csv")
air_check.check_range("C6H6(GT)", lower=0).df.show(5)

print("3. Only upper bound (Relative Humidity <= 80)")
air_check = SparkDataCheck.from_csv(spark, "air.csv")
air_check.check_range("RH", upper=80).df.show(5)

print("4. Non-numeric column (should print error message)")
air_check = SparkDataCheck.from_csv(spark, "air.csv")
air_check.check_range("Date", lower=0, upper=10).df.show(5)

print("5. Column with NULLs (NULLs preserved in result)")
air_check = SparkDataCheck.from_csv(spark, "air.csv")
air_check.check_range("CO(GT)", lower=-200, upper=10).df.show(5)

1. Both bounds on a numeric column (Temp between 0 and 40)
+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+----------+
|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|CO(GT)_in_range|T_in_range|
+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+---------------+----------+
|3/10/2004|2026-03-28 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|           true|      true|
|3/10/2004|2026-03-28 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|           true|      true|
|3/10/2004|2026-03-28 20:00:00|   2.

#### Check levels Validation
These validation checks test use cases and edge cases for upper and lower bound levels of a column

In [45]:
# check_levels examples

# 1. String column with matching values
air_check = SparkDataCheck.from_csv(spark, "air.csv")
print("1. Levels that exist in the data (Dates 10/03/2004, 11/03/2004, 12/03/2004)")
air_check.check_levels("Date", ["10/03/2004", "11/03/2004", "12/03/2004"]).df.show(25)
# TODO: show more rows to see all 3 dates, or just show the Date column

# 2. String column with values not in the data
air_check = SparkDataCheck.from_csv(spark, "air.csv")
print("2. Levels that do NOT exist in the data (Date 01/01/2099)")
air_check.check_levels("Date", ["01/01/2099"]).df.show(5)

# 3. Non-string column
air_check = SparkDataCheck.from_csv(spark, "air.csv")
print("3. Non-string column (should print error message)")
air_check.check_levels("T", ["hot", "cold"]).df.show(5)

1. Levels that exist in the data (Dates 10/03/2004, 11/03/2004, 12/03/2004)
+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------+
|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|Date_in_levels|
+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+--------------+
|3/10/2004|2026-03-28 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|         false|
|3/10/2004|2026-03-28 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|         false|
|3/10/2004|2026-03-28 20:00:00|   2.2|       1402|      88|     9.0|          9

#### Check Chaining of range and levels
Check that the ability to chain methods works between range and levels methods

In [41]:
# 4. Method chaining: check_range then check_levels
air_check = SparkDataCheck.from_csv(spark, "air.csv")
air_check.check_range("T", lower=10, upper=30).check_levels("Date", ["10/03/2004"]).df.show(5)

+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+----------+--------------+
|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|T_in_range|Date_in_levels|
+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+----------+--------------+
|3/10/2004|2026-03-28 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|      true|         false|
|3/10/2004|2026-03-28 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|      true|         false|
|3/10/2004|2026-03-28 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        11

#### Check Missing Values
TODO: Need to address that missing values are all -200

In [ ]:
# check_missing examples

# 1. Column likely to have NULLs
air_check = SparkDataCheck.from_csv(spark, "air.csv")
air_check.check_missing("NMHC(GT)").df.show(5)

# 2. Column unlikely to have NULLs
air_check = SparkDataCheck.from_csv(spark, "air.csv")
air_check.check_missing("Date").df.show(5)

# 3. Another numeric column
air_check = SparkDataCheck.from_csv(spark, "air.csv")
air_check.check_missing("CO(GT)").df.show(5)

# 4. Chained with check_range
air_check = SparkDataCheck.from_csv(spark, "air.csv")
air_check.check_missing("T").check_range("T", lower=0, upper=35).df.show(5)

#### Check min-max examples
These validation checks show the use and edge cases for obtaining min and max values with/without grouping variables

In [ ]:


# get_min_max examples

# 1. Single numeric column, no grouping
air_check = SparkDataCheck.from_csv(spark, "air.csv")
air_check.get_min_max("T")

# 2. Single numeric column, grouped by Date
air_check.get_min_max("T", group_col="Date")

# 3. All numeric columns, no grouping
air_check.get_min_max()

# 4. All numeric columns, grouped (reduce/merge case)
air_check.get_min_max(group_col="Date")

# 5. Non-numeric column — should print message, return None
air_check.get_min_max("Date")

# get_counts examples

# 1. Single string column
air_check = SparkDataCheck.from_csv(spark, "air.csv")
air_check.get_counts("Date")

# 2. Non-string column
air_check.get_counts("T")

# 3. Chained with check_levels
air_check = SparkDataCheck.from_csv(spark, "air.csv")
air_check.get_counts("Date", "T")

+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+----------+
|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|T_in_range|
+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+----------+
|3/10/2004|2026-03-28 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|      true|
|3/10/2004|2026-03-28 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|      true|
|3/10/2004|2026-03-28 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|      true|
|3/10/2004

### Demonstration of Summarization Methods

In [21]:
air_check = SparkDataCheck.from_csv(spark, "air.csv")

display(air_check.get_min_max("C6H6(GT)"))
display(air_check.get_counts("Date"))

,C6H6(GT)_min,C6H6(GT)_max
0,-200.0,63.7


,Date,count
0,1/1/2005,24
1,1/10/2005,24
2,1/11/2005,24
3,1/12/2005,24
4,1/13/2005,24
...,...,...
386,9/5/2004,24
387,9/6/2004,24
388,9/7/2004,24
389,9/8/2004,24


### Read in Air Quality Data (pandas method)

Now we read the same data using standard pandas and create an instance via `from_pandas`.

In [22]:
# Create instance from pandas DataFrame
air_pd = pd.read_csv("https://www4.stat.ncsu.edu/online/datasets/air.csv")
air_check_pd = SparkDataCheck.from_pandas(spark, air_pd)


In [23]:
# example method call on the pandas-created instance
air_check_pd.get_min_max("C6H6(GT)")

,C6H6(GT)_min,C6H6(GT)_max
0,-200.0,63.7


---

## Part II : NFL Quarterback Analysis

In this section we analyze weekly NFL data to look at quarterback performance across seasons 2005-2023. We perform the same analysis twice: once using the **pandas-on-Spark** API and once using the **Spark SQL** DataFrame API.

### pandas-on-Spark Approach

In [24]:
import pyspark.pandas as ps

# Read in the NFL data
nfl = ps.read_csv("weekly_nfl_data.csv")

# Check the first 5 rows
nfl.head(5)

/Users/david/projects/ncsu/st554/p2/.venv/lib/python3.12/site-packages/pyspark/pandas/utils.py:1038: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `read_csv`, the default index is attached which can cause additional overhead.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


,player_id,player_name,player_display_name,position,position_group,headshot_url,recent_team,season,week,season_type,opponent_team,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_2pt_conversions,pacr,dakota,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr
0,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,1,REG,DEN,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,16,60.0,1,0.0,0.0,4.0,6.248771,0,1,1,7.0,0,0.0,0.0,0.0,0.0,0.0,0.292378,0,0.0,0.052632,NaN,NaN,0.0,12.7,13.7
1,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,2,REG,ARI,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,9,33.0,0,0.0,0.0,1.0,-1.434950,0,3,4,18.0,0,0.0,0.0,0.0,0.0,1.0,0.377009,0,0.0,0.117647,NaN,NaN,0.0,5.1,8.1
2,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,4,REG,BUF,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,3,2.0,0,0.0,0.0,0.0,-1.539952,0,0,1,0.0,0,0.0,0.0,0.0,0.0,0.0,-0.699578,0,NaN,0.023810,NaN,NaN,0.0,0.2,0.2
3,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,7,REG,LA,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,6,27.0,0,0.0,0.0,0.0,0.216051,0,2,2,8.0,0,0.0,0.0,0.0,0.0,0.0,-0.228454,0,0.0,0.050000,NaN,NaN,0.0,3.5,5.5
4,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,8,REG,NO,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,13,39.0,0,0.0,0.0,2.0,-2.972259,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,3.9,3.9


In [25]:
# get columns
nfl.columns

Index(['player_id', 'player_name', 'player_display_name', 'position',
       'position_group', 'headshot_url', 'recent_team', 'season', 'week',
       'season_type', 'opponent_team', 'completions', 'attempts',
       'passing_yards', 'passing_tds', 'interceptions', 'sacks', 'sack_yards',
       'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards',
       'passing_yards_after_catch', 'passing_first_downs', 'passing_epa',
       'passing_2pt_conversions', 'pacr', 'dakota', 'carries', 'rushing_yards',
       'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost',
       'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions',
       'receptions', 'targets', 'receiving_yards', 'receiving_tds',
       'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards',
       'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa',
       'receiving_2pt_conversions', 'racr', 'target_share', 'air_yards_share',
       'wopr', 'special_teams_tds', 'fantasy_points

In [26]:
# Filter to QBs, regular season, 2005-2023
qb = nfl[(nfl["position"] == "QB") &
         (nfl["season_type"] == "REG") &
         (nfl["season"] >= 2005) &
         (nfl["season"] <= 2023)]

# Select only the columns we need
cols = ["player_display_name", "season", "week", "completions",
        "attempts", "passing_yards", "passing_tds", "interceptions"]
qb = qb[cols]

# GroupBy player and season, compute sum and mean
stat_cols = ["completions", "attempts", "passing_yards",
             "passing_tds", "interceptions"]
qb_season = qb.groupby(["player_display_name", "season"])[stat_cols].agg(["sum", "mean"])

# Flatten the MultiIndex columns
qb_season.columns = ["_".join(col) for col in qb_season.columns]
qb_season = qb_season.reset_index()

# Create new columns
qb_season["completion_percentage"] = qb_season["completions_sum"] / qb_season["attempts_sum"]
qb_season["td_int_ratio"] = qb_season["passing_tds_sum"] / qb_season["interceptions_sum"]



In [28]:
# Filter to at least 50 attempts
qualified = qb_season[qb_season["attempts_sum"] >= 50]

In [29]:
# Top 40 by completion percentage
qualified.sort_values("completion_percentage", ascending=False).head(40)

,player_display_name,season,completions_sum,completions_mean,attempts_sum,attempts_mean,passing_yards_sum,passing_yards_mean,passing_tds_sum,passing_tds_mean,interceptions_sum,interceptions_mean,completion_percentage,td_int_ratio
1409,C.J. Beathard,2023,40,6.666667,53,8.833333,349.0,58.166667,1,0.166667,0.0,0.000000,0.754717,inf
1248,Colt McCoy,2021,74,10.571429,99,14.142857,740.0,105.714286,3,0.428571,1.0,0.142857,0.747475,3.000000
900,Matt Schaub,2019,50,10.000000,67,13.400000,580.0,116.000000,3,0.600000,1.0,0.200000,0.746269,3.000000
953,Drew Brees,2018,364,24.266667,489,32.600000,3992.0,266.133333,32,2.133333,5.0,0.333333,0.744376,6.400000
1057,Drew Brees,2019,281,25.545455,378,34.363636,2979.0,270.818182,27,2.454545,4.0,0.363636,0.743386,6.750000
1338,Mason Rudolph,2023,55,13.750000,74,18.500000,719.0,179.750000,3,0.750000,0.0,0.000000,0.743243,inf
1133,Taysom Hill,2020,88,5.500000,121,7.562500,928.0,58.000000,4,0.250000,2.0,0.125000,0.727273,2.000000
1007,Nick Foles,2018,141,28.200000,195,39.000000,1413.0,282.600000,7,1.400000,4.0,0.800000,0.723077,1.750000
917,Drew Brees,2017,386,24.125000,536,33.500000,4334.0,270.875000,23,1.437500,8.0,0.500000,0.720149,2.875000
851,Sam Bradford,2016,395,26.333333,552,36.800000,3877.0,258.466667,20,1.333333,5.0,0.333333,0.715580,4.000000


### Top 40 TD / INT Ratio

In [30]:
# top 40 by TD/INT ratio
qualified.sort_values("td_int_ratio", ascending=False).head(40)

,player_display_name,season,completions_sum,completions_mean,attempts_sum,attempts_mean,passing_yards_sum,passing_yards_mean,passing_tds_sum,passing_tds_mean,interceptions_sum,interceptions_mean,completion_percentage,td_int_ratio
4,Kerry Collins,2007,50,8.333333,82,13.666667,531.0,88.500000,0,0.000000,0.0,0.000000,0.609756,NaN
829,Tom Savage,2016,46,15.333333,73,24.333333,461.0,153.666667,0,0.000000,0.0,0.000000,0.630137,NaN
941,Jacoby Brissett,2016,34,11.333333,55,18.333333,400.0,133.333333,0,0.000000,0.0,0.000000,0.618182,NaN
6,Charlie Batch,2006,30,4.285714,52,7.428571,477.0,68.142857,5,0.714286,0.0,0.000000,0.576923,inf
26,Matt Schaub,2005,33,4.714286,64,9.142857,495.0,70.714286,4,0.571429,0.0,0.000000,0.515625,inf
73,Todd Collins,2007,67,16.750000,105,26.250000,888.0,222.000000,5,1.250000,0.0,0.000000,0.638095,inf
455,Troy Smith,2007,40,10.000000,76,19.000000,452.0,113.000000,2,0.500000,0.0,0.000000,0.526316,inf
520,Jake Locker,2011,34,6.800000,66,13.200000,542.0,108.400000,4,0.800000,0.0,0.000000,0.515152,inf
775,Brian Hoyer,2016,134,22.333333,200,33.333333,1445.0,240.833333,6,1.000000,0.0,0.000000,0.670000,inf
778,Nick Foles,2016,36,18.000000,55,27.500000,410.0,205.000000,3,1.500000,0.0,0.000000,0.654545,inf


### Spark SQL DataFrame Approach

In [ ]:
# Read the NFL data using Spark SQL
nfl_spark = spark.read.load(
    "weekly_nfl_data.csv",
    format="csv",
    sep=",",
    inferSchema="true",
    header="true"
)

# Check first 5 rows
nfl_spark.show(5)

# get columns
nfl_spark.columns

+----------+-----------+--------------------+--------+--------------+------------+-----------+------+----+-----------+-------------+-----------+--------+-------------+-----------+-------------+-----+----------+------------+-----------------+-----------------+-------------------------+-------------------+-----------+-----------------------+----+------+-------+-------------+-----------+---------------+--------------------+-------------------+-----------+-----------------------+----------+-------+---------------+-------------+-----------------+----------------------+-------------------+---------------------------+---------------------+-------------+-------------------------+----+------------+---------------+----+-----------------+--------------+------------------+
| player_id|player_name| player_display_name|position|position_group|headshot_url|recent_team|season|week|season_type|opponent_team|completions|attempts|passing_yards|passing_tds|interceptions|sacks|sack_yards|sack_fumbles|sack_

['player_id',
 'player_name',
 'player_display_name',
 'position',
 'position_group',
 'headshot_url',
 'recent_team',
 'season',
 'week',
 'season_type',
 'opponent_team',
 'completions',
 'attempts',
 'passing_yards',
 'passing_tds',
 'interceptions',
 'sacks',
 'sack_yards',
 'sack_fumbles',
 'sack_fumbles_lost',
 'passing_air_yards',
 'passing_yards_after_catch',
 'passing_first_downs',
 'passing_epa',
 'passing_2pt_conversions',
 'pacr',
 'dakota',
 'carries',
 'rushing_yards',
 'rushing_tds',
 'rushing_fumbles',
 'rushing_fumbles_lost',
 'rushing_first_downs',
 'rushing_epa',
 'rushing_2pt_conversions',
 'receptions',
 'targets',
 'receiving_yards',
 'receiving_tds',
 'receiving_fumbles',
 'receiving_fumbles_lost',
 'receiving_air_yards',
 'receiving_yards_after_catch',
 'receiving_first_downs',
 'receiving_epa',
 'receiving_2pt_conversions',
 'racr',
 'target_share',
 'air_yards_share',
 'wopr',
 'special_teams_tds',
 'fantasy_points',
 'fantasy_points_ppr']

In [ ]:
from pyspark.sql import functions as F
# Filter and select
qb_spark = nfl_spark.filter(
    (F.col("position") == "QB") &
    (F.col("season_type") == "REG") &
    (F.col("season") >= 2005) &
    (F.col("season") <= 2023)
).select("player_display_name", "season", "week", "completions",
         "attempts", "passing_yards", "passing_tds", "interceptions")

# GroupBy and aggregate
qb_season_spark = qb_spark.groupBy("player_display_name", "season").agg(
    F.sum("completions").alias("completions_sum"),
    F.mean("completions").alias("completions_mean"),
    F.sum("attempts").alias("attempts_sum"),
    F.mean("attempts").alias("attempts_mean"),
    F.sum("passing_yards").alias("passing_yards_sum"),
    F.mean("passing_yards").alias("passing_yards_mean"),
    F.sum("passing_tds").alias("passing_tds_sum"),
    F.mean("passing_tds").alias("passing_tds_mean"),
    F.sum("interceptions").alias("interceptions_sum"),
    F.mean("interceptions").alias("interceptions_mean")
)

# Create new columns
# Use try_divide for td_int_ratio to handle QBs with 0 interceptions
qb_season_spark = qb_season_spark.withColumn(
    "completion_percentage",
    F.col("completions_sum") / F.col("attempts_sum")
).withColumn(
    "td_int_ratio",
    F.try_divide(F.col("passing_tds_sum"), F.col("interceptions_sum"))
)

# Filter to at least 50 attempts
qualified_spark = qb_season_spark.filter(F.col("attempts_sum") >= 50)

# Top 40 by completion percentage
qualified_spark.orderBy(F.col("completion_percentage").desc()).show(40)

# Top 40 by td_int_ratio
qualified_spark.orderBy(F.col("td_int_ratio").desc()).show(40)

+-------------------+------+---------------+------------------+------------+------------------+-----------------+------------------+---------------+-------------------+-----------------+-------------------+---------------------+------------------+
|player_display_name|season|completions_sum|  completions_mean|attempts_sum|     attempts_mean|passing_yards_sum|passing_yards_mean|passing_tds_sum|   passing_tds_mean|interceptions_sum| interceptions_mean|completion_percentage|      td_int_ratio|
+-------------------+------+---------------+------------------+------------+------------------+-----------------+------------------+---------------+-------------------+-----------------+-------------------+---------------------+------------------+
|      C.J. Beathard|  2023|             40| 6.666666666666667|          53| 8.833333333333334|            349.0|58.166666666666664|              1|0.16666666666666666|              0.0|                0.0|   0.7547169811320755|              NULL|
|       

### Comparison: pandas-on-Spark vs Spark SQL

#### Division by Zero in `td_int_ratio`

When computing `td_int_ratio = passing_tds_sum / interceptions_sum`, some 
quarterback-season combinations have zero interceptions, creating a division 
by zero problem. The two APIs handle this differently:

- **pandas-on-Spark** follows Python/NumPy conventions and produces a positive 
  infinity (`inf`) value. These values sort to the top when ordering descending, 
  so the top `td_int_ratio` results are dominated by QBs who happened to 
  throw zero interceptions in a season and seem to be players with minimal playing
  time.

- **Spark SQL** follows SQL conventions and produces `NULL`. These values 
  sort to the bottom when ordered by descending values ()`orderBy(...desc())`), so 
  those same QBs are effectively excluded from the top of the results. The Spark 
  SQL top 40 instead surfaces players who maintained higher ratios over the season.

This means the top 40 `td_int_ratio` lists look different between the 
two approaches with the default sort scenario, even though the underlying 
data and logic are identical.

For `completion_percentage`, the results are consistent across both APIs 
because our filter of `attempts_sum >= 50` guarantees a non-zero denominator. 
Players like Brock Purdy, who completed 69.4% of his passes during his 
breakout 2023 season (308 completions on 444 attempts with 31 touchdowns), 
appear in the same position in both outputs. (Faithful to The Bay!)

In [ ]:
# Stop Spark session
spark.stop()